# Sales Performance Analytics - Data Cleaning
**Step 1:** Load, explore, clean, and export the Superstore dataset.

In [ ]:
import pandas as pd
import numpy as np

# ── 1. LOAD ──────────────────────────────────────────────
df = pd.read_csv('Sample - Superstore.csv', encoding='latin-1')

print('Shape:', df.shape)
print('\nColumns:\n', df.columns.tolist())
df.head()

In [ ]:
# ── 2. BASIC EXPLORATION ─────────────────────────────────
print('--- Data Types ---')
print(df.dtypes)

print('\n--- Missing Values ---')
print(df.isnull().sum())

print('\n--- Duplicates ---')
print('Duplicate rows:', df.duplicated().sum())

In [ ]:
# ── 3. CLEAN ─────────────────────────────────────────────

# Fix date columns
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

# Drop duplicates
df = df.drop_duplicates()

# Strip whitespace from string columns
str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].apply(lambda x: x.str.strip())

# Rename columns: replace spaces with underscores, lowercase
df.columns = df.columns.str.strip().str.replace(' ', '_').str.lower()

print('Cleaned shape:', df.shape)
df.head()

In [ ]:
# ── 4. FEATURE ENGINEERING ───────────────────────────────

# Profit margin per order row
df['profit_margin_%'] = (df['profit'] / df['sales'] * 100).round(2)

# Shipping duration in days
df['shipping_days'] = (df['ship_date'] - df['order_date']).dt.days

# Year and Month for time-series
df['order_year']  = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month
df['order_month_name'] = df['order_date'].dt.strftime('%b')

# Revenue bucket (for segmentation)
df['revenue_bucket'] = pd.cut(
    df['sales'],
    bins=[0, 100, 500, 1000, 5000, 100000],
    labels=['Very Low', 'Low', 'Medium', 'High', 'Very High']
)

print('New columns added:')
print(['profit_margin_%', 'shipping_days', 'order_year', 'order_month', 'order_month_name', 'revenue_bucket'])
df.head()

In [ ]:
# ── 5. QUICK STATS ───────────────────────────────────────
print('=== Key Stats ===')
print(f"Total Orders  : {df['order_id'].nunique()}")
print(f"Total Customers: {df['customer_id'].nunique()}")
print(f"Total Sales   : ${df['sales'].sum():,.2f}")
print(f"Total Profit  : ${df['profit'].sum():,.2f}")
print(f"Avg Profit Margin: {df['profit_margin_%'].mean():.2f}%")
print(f"Date Range    : {df['order_date'].min().date()} to {df['order_date'].max().date()}")

In [ ]:
# ── 6. EXPORT CLEAN DATA ─────────────────────────────────
df.to_csv('superstore_cleaned.csv', index=False)
print('✅ Exported: superstore_cleaned.csv')
print('This file will be used in SQL and Power BI steps.')